In [1]:
import os
import json
import random
from openai import OpenAI
import anthropic
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm
import time


# Load dataset

In [2]:
# Set Paths
base_dir = os.path.join(os.getcwd(), "dataset", "pororo")
qa_json_path = os.path.join(base_dir, "qa.json")
description_csv_path = os.path.join(base_dir, "descriptions.csv")

def load_dataset(qa_json_path, description_csv_path):
    try:
        # Load QA data
        with open(qa_json_path, "r") as f:
            qa_data = json.load(f)["PororoQA"]
        
        # Load descriptions
        descriptions = pd.read_csv(description_csv_path)
        
        return qa_data, descriptions
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return [], pd.DataFrame()
    
def get_random_questions(qa_data, max_questions=20, base_pattern="Pororo_ENGLISH1", seed=42):
    """
    Get randomly distributed questions from Pororo episodes
    
    Args:
        qa_data: Full QA dataset
        max_questions: Maximum number of questions (default 20)
        base_pattern: Base pattern for filtering episodes
        seed: Random seed for reproducibility
    """
    random.seed(seed)
    
    # Filter all eligible questions
    filtered_questions = [q for q in qa_data if base_pattern in q["video_name"]]
    
    # Group by unique (video_name, supporting_num) pairs to avoid duplicates
    unique_pairs = {}
    for q in filtered_questions:
        key = (q["video_name"], q["supporting_num"]) 
        if key not in unique_pairs:
            unique_pairs[key] = []
        unique_pairs[key].append(q)
    
    # Sample from unique pairs
    unique_keys = list(unique_pairs.keys())
    selected_keys = random.sample(unique_keys, min(max_questions, len(unique_keys)))
    
    # Get one question from each selected pair
    sampled_questions = []
    episode_counts = {}
    
    for key in selected_keys:
        question = random.choice(unique_pairs[key])
        sampled_questions.append(question)
        episode = question["video_name"]
        episode_counts[episode] = episode_counts.get(episode, 0) + 1
    
    # Group by season for display
    season_episodes = {
        "Pororo_ENGLISH1_1": [],
        "Pororo_ENGLISH1_2": [],
        "Pororo_ENGLISH1_3": []
    }
    
    for ep in episode_counts.keys():
        season = "_".join(ep.split("_")[:3])
        if season in season_episodes:
            season_episodes[season].append(ep)
    
    # Print statistics
    print(f"\nSelected {len(sampled_questions)} questions from {len(episode_counts)} episodes:")
    for season in sorted(season_episodes.keys()):
        season_eps = {ep: episode_counts[ep] for ep in season_episodes[season]}
        if season_eps:
            print(f"\n{season}:")
            for ep in sorted(season_eps.keys()):
                print(f"  {ep}: {season_eps[ep]} questions")
    
    return sampled_questions

def get_seeded_question(questions, gif_num, base_seed=42):
    if not questions:
        return None
    # Create new Random instance for each GIF
    local_random = random.Random(base_seed + gif_num)
    # Sort questions to ensure consistent ordering
    sorted_questions = sorted(questions, key=lambda x: x["qid"])
    return local_random.choice(sorted_questions)

# Load data
qa_data, descriptions = load_dataset(qa_json_path, description_csv_path)

# Get random sample of questions
sampled_questions = get_random_questions(qa_data, max_questions=20)



Selected 20 questions from 13 episodes:

Pororo_ENGLISH1_1:
  Pororo_ENGLISH1_1_ep12: 2 questions
  Pororo_ENGLISH1_1_ep13: 2 questions
  Pororo_ENGLISH1_1_ep2: 3 questions
  Pororo_ENGLISH1_1_ep5: 1 questions
  Pororo_ENGLISH1_1_ep6: 3 questions
  Pororo_ENGLISH1_1_ep9: 1 questions

Pororo_ENGLISH1_2:
  Pororo_ENGLISH1_2_ep2: 1 questions
  Pororo_ENGLISH1_2_ep8: 1 questions

Pororo_ENGLISH1_3:
  Pororo_ENGLISH1_3_ep1: 1 questions
  Pororo_ENGLISH1_3_ep11: 1 questions
  Pororo_ENGLISH1_3_ep2: 1 questions
  Pororo_ENGLISH1_3_ep5: 2 questions
  Pororo_ENGLISH1_3_ep7: 1 questions


# Single agent prediction

In [3]:
load_dotenv()

# Configuration
MODEL_NAME = "gpt-4o-mini" 
# MODEL_NAME = "claude-3-5-haiku-20241022" 

# Determine which platform to use based on the model name
is_openai_model = not MODEL_NAME.startswith("claude-")

# Initialize appropriate client
if is_openai_model:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    print(f"Using OpenAI model: {MODEL_NAME}")
else:
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    print(f"Using Anthropic model: {MODEL_NAME}")

# Load descriptions
descriptions = pd.read_csv(description_csv_path)

def get_prediction(question, gif_paths, description, subtitles, max_retries=3, retry_delay=2):
    images = []
    for gif_path in gif_paths:
        with open(gif_path, "rb") as gif_file:
            images.append(gif_file.read())

    prompt = f"""
Please answer the following question based on the given context:

Question: {question}
Scene Description: {description}
Subtitles: {subtitles}

Guidelines for answering:
1. Focus on answering the specific question asked
2. Include all relevant details from the context
3. Maintain accuracy while being clear
4. Avoid unnecessary elaboration
5. Keep the answer complete but concise

Your answer should be factual and directly address the question.
"""
    for attempt in range(max_retries):
        try:
            if is_openai_model:
                # OpenAI implementation
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=150,
                    temperature=0.3,
                )
                return completion.choices[0].message.content.strip()
            else:
                # Anthropic implementation
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": prompt
                    }],
                    max_tokens=150,
                    temperature=0.3,
                )
                return completion.content[0].text.strip()

        except Exception as e:
            print(f"Prediction attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    print(f"All {max_retries} attempts failed for question: {question}")
    return None

Using OpenAI model: gpt-4o-mini


# Compute accuracy

In [4]:
def compute_accuracy(correct_answer, predicted_answer, max_retries=2, retry_delay=2):
    """
    Compare the correct answer with the predicted answer using semantic similarity
    Returns: float between 0 and 1 indicating the similarity/correctness
    """
    prompt = f"""
    Correct Answer: {correct_answer}
    Predicted Answer: {predicted_answer}

    Rate how well the predicted answer matches the correct answer on a scale of 0 to 1:
    - 1.0: Perfect match or completely correct meaning
    - 0.75: Mostly correct with minor differences
    - 0.5: Partially correct
    - 0.25: Slightly correct but missing key points
    - 0.0: Completely incorrect or unrelated

    Provide only the numeric score (e.g. 0.75) with no other text.
    """
    for attempt in range(max_retries):
        try:
            if is_openai_model:
                # OpenAI implementation
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=10,
                    temperature=0.3
                )
                score = float(completion.choices[0].message.content.strip())
            else:
                # Anthropic implementation
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": prompt
                    }],
                    max_tokens=10,
                    temperature=0.3
                )
                score = float(completion.content[0].text.strip())

            # Ensure score is between 0 and 1
            return max(0.0, min(1.0, score))

        except Exception as e:
            print(f"Accuracy calculation attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    # Return 0 if it can't parse the score
    return 0.0


# Evaluate model performance

In [5]:
# Create list to store evaluation results
evaluation_results = []

# Group questions by supporting_num
grouped_questions = {}
for entry in sampled_questions:
    video_name = entry["video_name"]
    supporting_num = entry["supporting_num"]
    key = (video_name, supporting_num)
    if key not in grouped_questions:
        grouped_questions[key] = []
    grouped_questions[key].append(entry)

# Get unique pairs to process
gif_pairs = sorted(list(grouped_questions.keys()))
correct_count = 0
total_count = len(gif_pairs)

# Process each video and GIF pair
for video_name, gif_num in tqdm(gif_pairs, total=total_count):
    current_questions = grouped_questions[(video_name, gif_num)]
    
    if not current_questions:
        print(f"No questions found for {video_name} GIF {gif_num}")
        continue
        
    # Use seeded random selection
    entry = get_seeded_question(current_questions, int(gif_num))
    
    # Get question info
    question = entry["question"]
    correct_idx = entry["correct_idx"]
    answers = [entry[f"answer{i}"] for i in range(5)]
    correct_answer = answers[correct_idx]
    qid = entry["qid"]

    # Construct paths dynamically based on video_name
    episode_parts = video_name.split("_")
    episode_folder = os.path.join(base_dir, "Scenes_Dialogues", 
                                "_".join(episode_parts[:-1]),
                                video_name)
    subtitles_path = os.path.join(episode_folder, "subtitles.txt")
    
    # Load subtitles
    with open(subtitles_path, "r") as f:
        subtitles = f.read()

    # Process current GIF
    gif_paths = [os.path.join(episode_folder, f"{gif_num}.gif")]

    # Get description
    description_row = descriptions.loc[descriptions.iloc[:, 0] == video_name]
    if description_row.empty:
        print(f"Description for {video_name} not found")
        continue
    description = description_row.iloc[0, 2]

    # Get prediction
    predicted_answer = get_prediction(question, gif_paths, description, subtitles)

    # Calculate accuracy
    is_correct = predicted_answer is not None and compute_accuracy(correct_answer, predicted_answer)
    correct_count += is_correct

    # Store current result
    result = {
        'gif_num': gif_num,
        'video_name': video_name,
        'qid': qid,
        'question': question,
        'correct_answer': correct_answer,
        'predicted_answer': predicted_answer,
        'accuracy': (is_correct)
    }
    evaluation_results.append(result)

    # Print debugging info
    print(f"\nVideo name: {video_name}")
    print(f"GIF number: {gif_num}")
    print(f"QID: {qid}")
    print(f"Question: {question}")
    print(f"Correct Answer: {correct_answer}")
    print(f"Predicted Answer: {predicted_answer}")
    print(f"Accuracy: {float(is_correct):.4f}")

# Calculate overall accuracy
average_accuracy = correct_count / total_count
print(f"\nAverage Accuracy: {average_accuracy:.4f}")

  5%|▌         | 1/20 [00:03<00:59,  3.11s/it]


Video name: Pororo_ENGLISH1_1_ep12
GIF number: 36
QID: 1222
Question: what does poby ask when he sees eddy
Correct Answer: poby asks eddy why is he so jumpy
Predicted Answer: Poby asks Eddy, "What is that box?" when he sees him.
Accuracy: 0.0000


 10%|█         | 2/20 [00:08<01:22,  4.59s/it]


Video name: Pororo_ENGLISH1_1_ep12
GIF number: 49
QID: 1232
Question: what does pororo say to crong after he realizes it was eddy and not crong
Correct Answer: pororo apologizes to crong and says he made a mistake
Predicted Answer: After Pororo realizes it was Eddy and not Crong who played the trick, he says to Crong, "I know that you did it," indicating his initial belief that Crong was responsible. However, after Eddy admits to placing the box and apologizes, Pororo also apologizes to Crong, leading to them making up and becoming good friends again.
Accuracy: 0.7500


 15%|█▌        | 3/20 [00:10<00:54,  3.23s/it]

...


Video name: Pororo_ENGLISH1_3_ep5
GIF number: 1
QID: 2296
Question: what were the friends talking about?
Correct Answer: the friends were talking about something secretly.
Predicted Answer: The friends, Eddy, Loopy, and Poby, were secretly talking about something that they didn't want Pororo to know. They were discussing a surprise for Pororo's birthday, as indicated by their secretive behavior and the eventual birthday wishes they gave him.
Accuracy: 1.0000


 95%|█████████▌| 19/20 [00:43<00:02,  2.26s/it]


Video name: Pororo_ENGLISH1_3_ep5
GIF number: 25
QID: 2334
Question: what did pororo's friends said?
Correct Answer: friends said : bye pororo.
Predicted Answer: Pororo's friends, Eddy, Loopy, and Poby, were secretly talking about something and initially tried to hide it from him. When Pororo asked what they were doing, they denied it and said "no nothing." Later, they revealed that it was Pororo's birthday and sang "happy birthday dear pororo happy birthday to you."
Accuracy: 0.2500


100%|██████████| 20/20 [00:45<00:00,  2.27s/it]


Video name: Pororo_ENGLISH1_3_ep7
GIF number: 25
QID: 2425
Question: what does crong do when pororo says "come here"
Correct Answer: crong runs away from pororo
Predicted Answer: When Pororo says "come here," Crong initially responds by making a series of "crong" sounds and then disappears, leading to Loopy's picture being ruined.
Accuracy: 0.2500

Average Accuracy: 0.4500


# Save data

In [6]:

# Remove any existing Average rows
evaluation_results = [r for r in evaluation_results if r['gif_num'] != 'Average']

# Get unique videos
unique_videos = len(set(r['video_name'] for r in evaluation_results))

# Add average accuracy as the last row
average_result = {
    'gif_num': 'Average',
    'video_name': f'Total Videos: {unique_videos}',
    'qid': '',
    'question': f'Total Questions: {len(evaluation_results)}',
    'correct_answer': '',
    'predicted_answer': '',
    'accuracy': average_accuracy
}
evaluation_results.append(average_result)

# Define column order
column_order = [
    'gif_num',
    'video_name', 
    'qid',
    'question',
    'correct_answer',
    'predicted_answer',
    'accuracy'
]

# Create safe model name for file naming
safe_model_name = MODEL_NAME.replace('-', '_').replace('.', '_')

# Set up output directory
results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)
output_path = os.path.join(
    results_dir,
    f'pororo_evaluation_results_single_agent_{safe_model_name}.csv'
)

# Remove existing file if it exists
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Existing file removed: {output_path}")
    except Exception as e:
        print(f"Error removing existing file: {e}")

# Save results with error handling
try:
    results_df = pd.DataFrame(evaluation_results)
    results_df = results_df[column_order]
    results_df.to_csv(output_path, index=False)
    
    if os.path.exists(output_path):
        print(f"Results successfully saved to: {output_path}")
    else:
        print(f"Warning: File was not created at {output_path}")
except Exception as e:
    print(f"Error saving results to CSV: {e}")

Results successfully saved to: /Users/wt/PythonProjects/MultimodalComicAgent/results/pororo_evaluation_results_single_agent_gpt_4o_mini.csv
